# Model Training Notebook

This notebook trains a Random Forest model on the gut survey data, builds a preprocessing pipeline, and saves the trained model.

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np
import re
import pickle
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge

## 1. Load and Inspect Data

In [2]:
# 1.1 Load your collected data
df = pd.read_csv("synthetic_bowel_movements.csv")
df.head()

,What is your age?,How would you rate the typical smell intensity of your stool?,"On average, how many hours of sleep do you get per night?",Timestamp,What is your gender?,Height (cm),Weight (kg),Hydration Level,How active are you physically on average?,Are you currently taking any medication that affects digestion or bowel movements?,...,How many meals with greasy or fried food do you eat per week?,"Do you regularly consume dairy products (milk, cheese, yogurt)?","On average, how many servings of processed food do you eat per day?","On average, how many servings of fruits and vegetables do you eat per day?",What type of toilet paper or wiping method do you usually use?,How would you describe your typical stool consistency?,What is the most common color of your stool?,"How many times do you go to the toilet for the number ""2"" in a week?","How many caffeinated beverages (coffee, tea, energy drinks) do you consume per day?","On average, how many wipes or sheets of toilet paper do you use per bowel movement?"
0,29,4,7,02/05/2025 22:48:43,Male,180,95,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,5,3-5,8
1,13,4,8,02/05/2025 23:02:38,Male,178,69,Moderate (1–2 liters/day),Low (mostly sedentary),No,...,0-2,Yes,0-2,0-2,3-ply paper,Soft,Brown,5-7,5 and more,24
2,20,2,6,02/05/2025 22:51:02,Male,161,74,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,5 and more,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,2-3,0-2,7
3,29,2,8,02/05/2025 23:02:48,Male,163,70,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,0-2,Yes,3-4,0-2,3-ply paper,Firm and smooth,Brown,14,0-2,Like 10 wipes
4,18,1,4,02/05/2025 22:50:20,Male,181,69,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,No,0-2,0-2,2-ply paper,Hard and lumpy,Brown,5,5 and more,3


In [3]:
column_mapping = {
    "What is your age?": "age",
    "How would you rate the typical smell intensity of your stool?": "smell_intensity",
    "On average, how many hours of sleep do you get per night?": "sleep_hours",
    "Timestamp": "timestamp",
    "What is your gender?": "gender",
    "Height (cm)": "height",
    "Weight (kg)": "weight",
    "Hydration Level": "hydration_level",
    "How active are you physically on average?": "activity_level",
    "Are you currently taking any medication that affects digestion or bowel movements?": "meds_affecting_gut",
    "How much dietary fibers do you eat daily?": "fiber_grams",
    "How much fat do you consume daily?": "fat_grams",
    "How spicy is your typical diet?": "spiciness",
    " How many meals with greasy or fried food do you eat per week?": "weekly_greasy_meals",
    "Do you regularly consume dairy products (milk, cheese, yogurt)?": "dairy_freq",
    "On average, how many servings of processed food do you eat per day?": "processed_servings",
    "On average, how many servings of fruits and vegetables do you eat per day?": "fv_servings",
    "What type of toilet paper or wiping method do you usually use?": "toilet_method",
    "How would you describe your typical stool consistency?": "stool_consistency",
    "What is the most common color of your stool?": "stool_color",
    "How many times do you go to the toilet for the number \"2\" in a week?": "weekly_bms",
    "How many caffeinated beverages (coffee, tea, energy drinks) do you consume per day?": "caffeinated_beverages_per_day",
    "On average, how many wipes or sheets of toilet paper do you use per bowel movement?": "wipes_per_bm"
}

   

   
# Apply the mapping
df.rename(columns=column_mapping, inplace=True)
df.head()

,age,smell_intensity,sleep_hours,timestamp,gender,height,weight,hydration_level,activity_level,meds_affecting_gut,...,weekly_greasy_meals,dairy_freq,processed_servings,fv_servings,toilet_method,stool_consistency,stool_color,weekly_bms,caffeinated_beverages_per_day,wipes_per_bm
0,29,4,7,02/05/2025 22:48:43,Male,180,95,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,5,3-5,8
1,13,4,8,02/05/2025 23:02:38,Male,178,69,Moderate (1–2 liters/day),Low (mostly sedentary),No,...,0-2,Yes,0-2,0-2,3-ply paper,Soft,Brown,5-7,5 and more,24
2,20,2,6,02/05/2025 22:51:02,Male,161,74,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,5 and more,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,2-3,0-2,7
3,29,2,8,02/05/2025 23:02:48,Male,163,70,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,0-2,Yes,3-4,0-2,3-ply paper,Firm and smooth,Brown,14,0-2,Like 10 wipes
4,18,1,4,02/05/2025 22:50:20,Male,181,69,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,No,0-2,0-2,2-ply paper,Hard and lumpy,Brown,5,5 and more,3


## 2. Pipeline Creation

In [14]:
from typing import List, Optional, Tuple, cast
from sklearn.base import RegressorMixin, BaseEstimator
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import FeatureUnion, FunctionTransformer, make_pipeline

def filter_columns(
    X: pd.DataFrame, columns: Optional[List[str]] = None
) -> pd.DataFrame:
    """A function that filters the columns of the input data.

    Args:
        X (pd.DataFrame): The input data.
        columns (Optional[List[str]], optional): The columns that should be kept. Defaults to None.

    Returns:
        pd.DataFrame: The input data with only the specified columns.
    """

    if columns is not None:
        return X[columns]

    return X


def clean_short_open_numerical_cols(y: pd.Series) -> pd.Series:
    # replace digit-digit with mean of those ranges
    # set not-numeric values to NaN
    
    y = pd.to_numeric(y, errors="coerce")

    return y

def clean_data(X: pd.DataFrame) -> pd.DataFrame:
    r"""
    numeric_cols = ['age', 'height', 'weight', 'sleep_hours']
for col in numeric_cols:
    # strip non‐digits (e.g. '90kg = bulking' → '90')
    df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)


# 6. Helper to parse ranges like "3-4", "5 and more", "0-2"
def parse_range(val):
    if pd.isna(val): 
        return np.nan
    s = str(val)
    if 'and more' in s:
        return float(re.search(r'(\d+)', s).group(1))
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    # fallback: extract single number
    m2 = re.search(r'(\d+)', s)
    return float(m2.group(1)) if m2 else np.nan

# 7. Apply range parser to these columns
range_cols = [
    'smell_intensity', 'weekly_greasy_meals', 'processed_servings', 
    'fv_servings', 'weekly_bms', 'wipes_per_bm'
]
for col in range_cols:
    df[col] = df[col].apply(parse_range)

# 1. Compute the mean of weekly bms excluding 100
mean_bms = df.loc[df['weekly_bms'] != 100, 'weekly_bms'].mean()
df.loc[df['weekly_bms'] == 100, 'weekly_bms'] = mean_bms

# Map Male→0, Female→1, leave Other as NaN for now
df['gender_encoded'] = df['gender'].map({'Male': 0, 'Female': 1})

# Randomly assign 0 or 1 for the rows where gender == 'Other'
mask_other = df['gender'] == 'Other'
df.loc[mask_other, 'gender_encoded'] = np.random.randint(0, 2, size=mask_other.sum())

# Drop the original gender column if you no longer need it
df.drop(['gender', 'timestamp', 'meds_affecting_gut'], axis=1, inplace=True)

# 2. One-hot encode the remaining categorical columns
categorical_cols = [
    'hydration_level',
    'activity_level',
    'dairy_freq',
    'spiciness',
    'toilet_method',
    'stool_consistency',
    'stool_color',
    'caffeinated_beverages_per_day',
    'fat_grams',
    'fiber_grams'
]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

    """
    X['height'] = X['height'].astype(str).str.replace(r'[,\.]', '', regex=True).astype(float)
    X['weight'] = pd.to_numeric(X['weight'], errors='coerce')

    short_open_numerical_cols = [
        'weekly_bms', 
        'wipes_per_bm'
    ]

    for col in short_open_numerical_cols:
        if col in X.columns:
            X[col] = clean_short_open_numerical_cols(X[col])

    return X


def create_pipeline(estimator: RegressorMixin, features_in: List[str], pandas_output: bool = False) -> Pipeline:

    """
    select faetures
    clean up (bugs and typos in the data)
    impute missing values
    standardize numerical 
    encode categorical
    """
    main_pipeline_steps : List[Tuple[str, BaseEstimator]] = [
        (
            "data_cleaning",
            FunctionTransformer(
                clean_data,
                validate=False,
            ),
        ),
        (
            "column_selector",
            FunctionTransformer(
                filter_columns,
                kw_args={"columns": features_in},
                validate=False,
            ),
        ),
        (
            "feature_union",
            FeatureUnion(
                transformer_list=[
                    (
                        "numerical",
                        ColumnTransformer(
                            transformers=[
                                (
                                    "numerical_pipeline",
                                    make_pipeline(
                                        SimpleImputer(strategy="mean"),
                                        StandardScaler(),
                                    ),
                                    make_column_selector(dtype_include=np.number),  # type: ignore
                                ),
                            ],
                            remainder="drop",
                        ),
                    ),
                    (
                        "string",
                        ColumnTransformer(
                            transformers=[
                                (
                                    "string_pipeline",
                                    make_pipeline(
                                        SimpleImputer(strategy="constant", fill_value="dna"),
                                        OneHotEncoder(
                                            drop="first",
                                            handle_unknown="infrequent_if_exist",
                                            sparse_output=not pandas_output,
                                        ),
                                    ),
                                    make_column_selector(dtype_include=object),  # type: ignore
                                ),
                            ],
                            remainder="drop",
                        ),
                    ),
                ]
            ),
        )
    ]


    if estimator is not None:
        main_pipeline_steps.append(
            (
                "estimator",
                estimator,
            )
        )

    pipeline = cast(
        Pipeline,
        Pipeline(steps=main_pipeline_steps).set_output(
            transform="pandas" if pandas_output else "default"
        ),
    )    
    return pipeline
    

class CreateYPipeline:
    def __init__(self, lower_bound: float, upper_bound: float):
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound

        self.lower_value = None
        self.upper_value = None

    def fit(self, y: pd.Series) -> "CreateYPipeline":
        self.lower_value = np.percentile(y, self.lower_bound)
        self.upper_value = np.percentile(y, self.upper_bound)
        return self
    
    def transform(self, y: pd.Series) -> pd.Series:
        y = y.copy()
        y.loc[y < self.lower_value] = self.lower_value
        y.loc[y > self.upper_value] = self.upper_value
        return y
    
    def fit_transform(self, y: pd.Series) -> pd.Series:
        self.fit(y)
        return self.transform(y)
    
    def print(self):
        print(f"Lower bound: {self.lower_value}")
        print(f"Upper bound: {self.upper_value}")
        print(f"Lower bound percentile: {self.lower_bound}")
        print(f"Upper bound percentile: {self.upper_bound}")
    
    


## 3. Data splitting

In [23]:
from sklearn.model_selection import train_test_split

X_raw = df.drop(columns=['wipes_per_bm']).copy()
y_raw = df['wipes_per_bm'].copy()

y = pd.to_numeric(y_raw.copy().loc[~clean_short_open_numerical_cols(y_raw).isna()])
X = X_raw.copy().loc[y.index]

X_train, X_test, y_train_raw, y_test_raw = train_test_split(X, y, test_size=0.2, random_state=42)

y_pipeline = CreateYPipeline(20, 80)

y_train = y_pipeline.fit_transform(y_train_raw)
y_test = y_pipeline.transform(y_test_raw)




y_pipeline.print()

Lower bound: 3.0
Upper bound: 12.0
Lower bound percentile: 20
Upper bound percentile: 80


## 4. Model Evaluation

In [24]:

pipeline = create_pipeline(
    estimator=Ridge(),
    features_in=X.columns.tolist(),
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# have the rmse evaluation
from sklearn.metrics import mean_squared_error
from math import sqrt
rmse = sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.2f}")

RMSE: 3.53


## 5. Save the Trained Model

In [26]:

pipeline = create_pipeline(
    estimator=Ridge(),
    features_in=X.columns.tolist(),
)
pickle.dump(pipeline, open("estimator.pkl",'wb'))
print("Model trained and saved to model.pkl")

Model trained and saved to model.pkl
